# Day 8 — One query vs many documents

**Milestone 1:** a real corpus has thousands of documents. Today we score a query against **all of them at once** — no Python loop — which is the search scan inside RAG.

## 1. Review (~5 min)

Recall **day 1** (Σ linearity) and **day 5** (cosine of the angle).

In [ ]:
import numpy as np
rng = np.random.default_rng(8)

**R1.** (day 1) $\sum_i (a\,x_i + b\,y_i)$.

In [ ]:
def r_combo_sum(a, b, x, y):
    raise NotImplementedError("# YOUR CODE HERE")

In [ ]:
def check_r1(fn):
    for _ in range(4):
        n = int(rng.integers(1, 8)); a = rng.normal(); b = rng.normal()
        x = rng.normal(size=n); y = rng.normal(size=n)
        assert np.allclose(fn(a, b, x, y), a*np.sum(x) + b*np.sum(y))
    print("✅ Review R1 passed")

check_r1(r_combo_sum)

**R2.** (day 5) $\cos\theta = a\cdot b/(\lVert a\rVert\lVert b\rVert)$.

In [ ]:
def r_cos(a, b):
    raise NotImplementedError("# YOUR CODE HERE")

In [ ]:
def check_r2(fn):
    for _ in range(4):
        n = int(rng.integers(2, 8)); a = rng.normal(size=n); b = rng.normal(size=n)
        assert np.allclose(fn(a, b), np.dot(a, b)/(np.linalg.norm(a)*np.linalg.norm(b)))
    print("✅ Review R2 passed")

check_r2(r_cos)

## 2. Lecture note (~10 min): scoring a whole corpus

**The itch.** Computing cosine similarity to a million documents with a Python `for` loop is painfully slow. We want every score in one shot.

**The picture.** Stack the documents as the **rows** of a matrix $D$ of shape $(n, d)$; the query is a vector $q$ of shape $(d,)$. We want a length-$n$ vector whose $i$-th entry is $\operatorname{cossim}(q, D_i)$.

**The formula.** Normalize the query, $\hat q$, and each row, $\hat D_i$. Then every cosine score is a dot product, and *all* of them together are a single **matrix–vector product**:
$$\text{scores} = \hat D\,\hat q, \qquad \text{scores}_i = \hat D_i\cdot\hat q.$$
Matrix multiply is just "a batch of dot products" — exactly the leap we'll reuse for $QK^\top$ in attention.

**Knobs & effect.** The win is **vectorization**: replace the loop with one `@`. Output shape is $(n,)$, one score per document. (If you pre-normalize the stored corpus once, each query is then just $D_{\text{norm}}\,\hat q$.)

**In the wild.** This is the brute-force similarity scan at the core of a vector database / RAG retriever (before fancy approximate-nearest-neighbor indexing).

## 3. Exercises (~15 min)

### Exercise 1 — cosine similarity to every row
Given query `q` (shape `(d,)`) and corpus `D` (shape `(n, d)`), return a length-`n` array of cosine similarities. (Checked against a per-row loop oracle.)

In [ ]:
def cosine_sim_to_corpus(q, D):
    raise NotImplementedError("# YOUR CODE HERE")

In [ ]:
def check_e1(fn):
    for _ in range(4):
        n = int(rng.integers(2, 6)); d = int(rng.integers(2, 6))
        q = rng.normal(size=d); D = rng.normal(size=(n, d))
        out = np.asarray(fn(q, D))
        assert out.shape == (n,)
        oracle = np.array([np.dot(q, D[i])/(np.linalg.norm(q)*np.linalg.norm(D[i])) for i in range(n)])
        assert np.allclose(out, oracle)
    print("✅ Exercise 1 passed")

check_e1(cosine_sim_to_corpus)

### Exercise 2 — the vectorized form (the effect)
Compute the same scores as **one matrix–vector product** of normalized rows with the normalized query: $\hat D\,\hat q$. Confirm it matches the loop oracle (i.e. no loop needed).

In [ ]:
def scores_vectorized(q, D):
    raise NotImplementedError("# YOUR CODE HERE")

In [ ]:
def check_e2(fn):
    for _ in range(4):
        n = int(rng.integers(2, 6)); d = int(rng.integers(2, 6))
        q = rng.normal(size=d); D = rng.normal(size=(n, d))
        oracle = np.array([np.dot(q, D[i])/(np.linalg.norm(q)*np.linalg.norm(D[i])) for i in range(n)])
        assert np.allclose(np.asarray(fn(q, D)), oracle)
    print("✅ Exercise 2 passed")

check_e2(scores_vectorized)

### Exercise 3 — the single best match
Return the **index** of the document most similar to `q` (highest cosine). The check plants the query as a copy of one document and expects that one back.

In [ ]:
def which_doc(q, D):
    raise NotImplementedError("# YOUR CODE HERE")

In [ ]:
def check_e3(fn):
    n, d = 5, 4
    D = rng.normal(size=(n, d))
    q = D[2].copy()  # identical to document 2
    assert fn(q, D) == 2
    print("✅ Exercise 3 passed")

check_e3(which_doc)

## Solutions (collapsed — peek only if stuck)

`ref_`-prefixed answers. Running the next cell validates everything against the checks above.

In [ ]:
def ref_combo_sum(a, b, x, y):
    return a*np.sum(x) + b*np.sum(y)

def ref_cos(a, b):
    return np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b))

def ref_cosine_sim_to_corpus(q, D):
    qn = q / np.linalg.norm(q)
    Dn = D / np.linalg.norm(D, axis=1, keepdims=True)
    return Dn @ qn

def ref_scores_vectorized(q, D):
    return ref_cosine_sim_to_corpus(q, D)

def ref_which_doc(q, D):
    return int(np.argmax(ref_cosine_sim_to_corpus(q, D)))

check_r1(ref_combo_sum)
check_r2(ref_cos)
check_e1(ref_cosine_sim_to_corpus)
check_e2(ref_scores_vectorized)
check_e3(ref_which_doc)
print("\nAll reference solutions pass. 🎉  See you on day 9!")